# Divide-and-Conquer: Merge Sort and Recurrence Analysis

**Session:** 2026-11-08 · Dynamic Programming and Divide-and-Conquer  
**Module:** 01 · Python Fundamentals and Algorithmic Thinking  

---

## 1. Algorithmic Overview: The Divide-and-Conquer Paradigm

Divide-and-Conquer is an algorithmic paradigm based on multi-branched recursion:
1. **Divide:** Decompose the input problem instance of size $n$ into $a$ smaller, independent subproblems of size $n/b$.
2. **Conquer:** Solve each subproblem recursively. When the subproblem size is sufficiently small ($n \le n_0$), solve it directly as a base case.
3. **Combine:** Assemble the solutions of the subproblems into the solution for the original instance.

In this notebook, we examine **Merge Sort**, formulate its recurrence relation, analyze its execution tree, and prove its $O(n \log n)$ time complexity using the Master Theorem.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

DATA_DIR = Path("../data")
with (DATA_DIR / "sorting_benchmarks.json").open(encoding="utf-8") as f:
    benchmarks = json.load(f)

print(f"Loaded {len(benchmarks)} sorting benchmark cases.")

## 2. Merge Procedure and Recursive Merge Sort Implementation

The merge procedure combines two pre-sorted sequences $S_1$ and $S_2$ into an aggregate array $S$. By comparing head pointers $i$ and $j$, the minimal available element is copied in $O(|S_1| + |S_2|)$ time.


In [ ]:
def merge(s1: list[int], s2: list[int], s: list[int]) -> None:
    """Merge two sorted sequences s1 and s2 into sequence s in-place."""
    i = j = 0
    total_len = len(s)
    len1 = len(s1)
    len2 = len(s2)

    while i + j < total_len:
        if j == len2 or (i < len1 and s1[i] <= s2[j]):
            s[i + j] = s1[i]
            i += 1
        else:
            s[i + j] = s2[j]
            j += 1


def merge_sort(s: list[int]) -> None:
    """Sort a sequence s using recursive Divide-and-Conquer Merge Sort."""
    n = len(s)
    if n < 2:
        return

    mid = n // 2
    s1 = s[:mid]
    s2 = s[mid:]

    merge_sort(s1)
    merge_sort(s2)

    merge(s1, s2, s)


# Verify on the primary lecture sequence
sample_array = benchmarks["lecture_example"]["array"].copy()
print("Original sequence:", sample_array)
merge_sort(sample_array)
print("Sorted sequence:  ", sample_array)
assert sample_array == sorted(benchmarks["lecture_example"]["array"])

## 3. Execution Tracing and Call Tree Visualization

To visualize the exact Divide, Conquer, and Combine steps executed during Merge Sort, we implement a diagnostic execution tracer.


In [ ]:
def merge_sort_traced(arr: list[int], depth: int = 0) -> list[int]:
    """Divide-and-conquer merge sort with execution logging."""
    indent = "  " * depth
    n = len(arr)
    if n <= 1:
        print(f"{indent}-> Base case reached: {arr}")
        return arr

    mid = n // 2
    print(f"{indent}[DIVIDE] Splitting array of length {n}: {arr[:mid]} | {arr[mid:]}")
    left = merge_sort_traced(arr[:mid], depth + 1)
    right = merge_sort_traced(arr[mid:], depth + 1)

    merged: list[int] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    print(f"{indent}[COMBINE] Merged {left} and {right} -> {merged}")
    return merged


lecture_input = benchmarks["lecture_example"]["array"]
print("=== Execution Trace for Lecture Input ===")
result = merge_sort_traced(lecture_input)
assert result == sorted(lecture_input)

## 4. Recurrence Analysis and The Master Theorem

The computational cost of Merge Sort satisfies:
$$T(n) = \begin{cases} O(1) & \text{if } n \le 1 \\ 2 T(n/2) + cn & \text{if } n > 1 \end{cases}$$

Where:
- $a = 2$: Two recursive subproblems.
- $b = 2$: Subproblem size is halved ($n/2$).
- $f(n) = cn = \Theta(n^1)$: Linear time to divide ($O(1)$) and merge ($O(n)$).

Applying the Master Theorem ($T(n) = a T(n/b) + f(n)$):
$$\log_b a = \log_2 2 = 1 \implies n^{\log_b a} = n^1$$
Since $f(n) = \Theta(n^1) = \Theta(n^{\log_b a})$, **Case 2** applies:
$$T(n) = \Theta(n^{\log_b a} \log n) = \Theta(n \log n)$$

At every level $k$ of the recursion tree (from $k = 0$ to $\log_2 n$), there are $2^k$ subproblems, each doing $c \cdot (n / 2^k)$ work. The sum per level is exactly $c n$. Across $\log_2 n + 1$ levels, total cost is:
$$\sum_{k=0}^{\log_2 n} c n = c n (\log_2 n + 1) = \Theta(n \log n)$$


## 5. Automated Validation on Full Benchmark Suite


In [ ]:
for name, case in benchmarks.items():
    test_arr = case["array"].copy()
    expected = sorted(test_arr)
    merge_sort(test_arr)
    assert test_arr == expected, f"Failed case {name}: got {test_arr}, expected {expected}"
    print(f"PASS: {name} (length {len(test_arr)})")